Yes. Python can do both, but they are slightly different:

* **tcpdump-like capture** → capture packets and inspect TCP/UDP/IP headers.
* **Port scanning** → test which TCP ports on a host are accepting connections.

Only scan/capture systems and networks you own or are authorized to test.

### 1. Simple TCP port scanner with Python

This uses only Python's built-in `socket` module:

```python
import socket

target = "127.0.0.1"

ports = [22, 53, 80, 443, 3389, 8080]

print(f"Scanning: {target}")

for port in ports:
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(1)

    result = sock.connect_ex((target, port))

    if result == 0:
        print(f"TCP {port}: OPEN")
    else:
        print(f"TCP {port}: CLOSED/FILTERED")

    sock.close()
```

You can change:

```python
target = "127.0.0.1"
```

to an authorized lab device, for example:

```python
target = "10.0.0.10"
```

### 2. Scan a range of TCP ports

For example, ports **1–1024**:

```python
import socket

target = "127.0.0.1"

for port in range(1, 1025):

    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(0.2)

    result = sock.connect_ex((target, port))

    if result == 0:
        print(f"[OPEN] TCP/{port}")

    sock.close()
```

This is a **TCP connect scan**. It completes the normal TCP connection rather than sending raw SYN packets.

### 3. Python equivalent of a basic `tcpdump`

For packet capture, **Scapy** is useful.

Install:

```powershell
pip install scapy
```

Then:

```python
from scapy.all import sniff

def packet_received(packet):
    print(packet.summary())

print("Starting packet capture...")

sniff(
    filter="tcp",
    prn=packet_received,
    store=False,
    count=20
)
```

This is conceptually similar to:

```bash
tcpdump tcp
```

On **Windows**, you will normally also need **Npcap** and may need to run PowerShell/Terminal as Administrator.

### 4. Capture traffic for a particular TCP port

For HTTPS/TCP 443:

```python
from scapy.all import sniff

def packet_received(packet):
    print(packet.summary())

sniff(
    filter="tcp port 443",
    prn=packet_received,
    store=False,
    count=50
)
```

Equivalent idea:

```bash
tcpdump tcp port 443
```

You can also filter by host:

```python
sniff(
    filter="host 8.8.8.8 and tcp",
    prn=lambda p: print(p.summary()),
    store=False,
    count=20
)
```

### 5. Save the capture as a `.pcap` file

This is particularly useful for network engineering because you can open the resulting file in Wireshark:

```python
from scapy.all import sniff, wrpcap

print("Capturing packets...")

packets = sniff(
    filter="tcp",
    count=100
)

wrpcap("capture.pcap", packets)

print("Saved as capture.pcap")
```

Then open:

```text
capture.pcap
```

in Wireshark and inspect:

**Ethernet → IP → TCP → Application protocol**

### Useful progression for network automation

A good learning sequence would be:

```text
Python socket
    ↓
TCP connect test
    ↓
TCP port scanner
    ↓
Scapy packet capture
    ↓
TCP SYN/SYN-ACK analysis
    ↓
PCAP generation
    ↓
Wireshark analysis
    ↓
Automated network troubleshooting
```

For example, we can build a Python troubleshooting script that takes `8.8.8.8:443` and automatically reports **DNS resolution → ping → TCP connection → latency → local/source IP → TCP packet capture → PCAP file**, which is much closer to what you'd use as a network engineer.


In [2]:
import socket

target = "127.0.0.1"

ports = [22, 53, 80, 443, 3389, 8080]

print(f"Scanning: {target}")

for port in ports:
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(1)

    result = sock.connect_ex((target, port))

    if result == 0:
        print(f"TCP {port}: OPEN")
    else:
        print(f"TCP {port}: CLOSED/FILTERED")

    sock.close()

Scanning: 127.0.0.1
TCP 22: CLOSED/FILTERED
TCP 53: CLOSED/FILTERED
TCP 80: CLOSED/FILTERED
TCP 443: CLOSED/FILTERED
TCP 3389: OPEN
TCP 8080: CLOSED/FILTERED


In [3]:
import socket

target = "127.0.0.1"

for port in range(1, 1025):

    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(0.2)

    result = sock.connect_ex((target, port))

    if result == 0:
        print(f"[OPEN] TCP/{port}")

    sock.close()

[OPEN] TCP/135
[OPEN] TCP/445
[OPEN] TCP/902
[OPEN] TCP/912


In [4]:
!pip install scapy

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://packagefeedproxy.microsoft.io/pypi/simple/



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\v-yaalam\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
from scapy import sniff

def packet_received(packet):
    print(packet.summary())

print("Starting packet capture...")

sniff(
    filter="tcp",
    prn=packet_received,
    store=False,
    count=20
)

ModuleNotFoundError: No module named 'scapy'

In [8]:
from scapy.all import sniff, wrpcap

print("Capturing packets...")

packets = sniff(
    filter="tcp",
    count=100
)

wrpcap("capture.pcap", packets)

print("Saved as capture.pcap")

ModuleNotFoundError: No module named 'scapy'

Here’s a practical **Python network troubleshooting script** that checks DNS, ping, TCP connectivity, latency, source IP, and can optionally capture packets to a PCAP file.

```python
import socket
import subprocess
import platform
import time
from datetime import datetime

TARGET = "8.8.8.8"
PORT = 443
TIMEOUT = 3


def resolve_dns(target):
    print("\n=== DNS Resolution ===")

    try:
        ip = socket.gethostbyname(target)
        print(f"Target : {target}")
        print(f"IP     : {ip}")
        return ip

    except socket.gaierror as error:
        print(f"DNS resolution failed: {error}")
        return None


def ping_test(target):
    print("\n=== Ping Test ===")

    system = platform.system().lower()

    if system == "windows":
        command = ["ping", "-n", "4", target]
    else:
        command = ["ping", "-c", "4", target]

    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True
        )

        print(result.stdout)

        if result.returncode == 0:
            print("Ping: SUCCESS")
        else:
            print("Ping: FAILED")

    except Exception as error:
        print(f"Ping error: {error}")


def tcp_test(target, port):
    print("\n=== TCP Connectivity Test ===")

    sock = socket.socket(
        socket.AF_INET,
        socket.SOCK_STREAM
    )

    sock.settimeout(TIMEOUT)

    start_time = time.time()

    try:
        result = sock.connect_ex((target, port))

        elapsed_ms = (time.time() - start_time) * 1000

        if result == 0:
            print(f"TCP/{port}: OPEN")
            print(f"Connection time: {elapsed_ms:.2f} ms")

            local_ip, local_port = sock.getsockname()

            print(f"Source IP   : {local_ip}")
            print(f"Source Port : {local_port}")
            print(f"Destination : {target}:{port}")

            return True

        else:
            print(f"TCP/{port}: CLOSED or FILTERED")
            print(f"Socket error code: {result}")

            return False

    except socket.timeout:
        print(f"TCP/{port}: TIMEOUT")

    except Exception as error:
        print(f"TCP test error: {error}")

    finally:
        sock.close()

    return False


def show_route_info(target):
    print("\n=== Route Information ===")

    system = platform.system().lower()

    if system == "windows":
        command = ["tracert", "-d", target]
    else:
        command = ["traceroute", "-n", target]

    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True
        )

        print(result.stdout)

    except Exception as error:
        print(f"Traceroute error: {error}")


def main():

    print("=" * 60)
    print("PYTHON NETWORK TROUBLESHOOTING TOOL")
    print("=" * 60)

    print(f"Time   : {datetime.now()}")
    print(f"Target : {TARGET}")
    print(f"Port   : {PORT}")

    ip = resolve_dns(TARGET)

    if ip is None:
        return

    ping_test(ip)

    tcp_test(ip, PORT)

    show_route_info(ip)

    print("\n" + "=" * 60)
    print("Troubleshooting completed")
    print("=" * 60)


if __name__ == "__main__":
    main()
```

For your example:

```python
TARGET = "8.8.8.8"
PORT = 443
```

You might see output similar to:

```text
============================================================
PYTHON NETWORK TROUBLESHOOTING TOOL
============================================================

Target : 8.8.8.8
Port   : 443

=== DNS Resolution ===
Target : 8.8.8.8
IP     : 8.8.8.8

=== Ping Test ===

Reply from 8.8.8.8: bytes=32 time=10ms TTL=118
Reply from 8.8.8.8: bytes=32 time=11ms TTL=118
Reply from 8.8.8.8: bytes=32 time=10ms TTL=118

Ping: SUCCESS

=== TCP Connectivity Test ===

TCP/443: OPEN
Connection time: 15.42 ms

Source IP   : 10.0.9.200
Source Port : 51432
Destination : 8.8.8.8:443

=== Route Information ===

Tracing route to 8.8.8.8...

1    <1 ms    10.0.9.1
2     3 ms    ...
3     8 ms    ...
```

This is basically a Python version of combining:

```powershell
nslookup 8.8.8.8

ping 8.8.8.8

Test-NetConnection 8.8.8.8 -Port 443

tracert 8.8.8.8
```

For packet capture, install Scapy:

```powershell
pip install scapy
```

On Windows, also install **Npcap**. Then you can add this:

```python
from scapy.all import sniff, wrpcap


def capture_packets(target, port):

    print("\n=== Packet Capture ===")

    packet_filter = f"host {target} and tcp port {port}"

    print(f"Filter: {packet_filter}")
    print("Capturing 20 packets...")

    packets = sniff(
        filter=packet_filter,
        count=20,
        timeout=20
    )

    filename = f"capture_{target}_{port}.pcap"

    wrpcap(filename, packets)

    print(f"Captured packets: {len(packets)}")
    print(f"Saved to: {filename}")
```

Then call:

```python
capture_packets("8.8.8.8", 443)
```

The PCAP can be opened directly in **Wireshark**.

A particularly useful next version for network engineering is to make it behave like this:

```text
python network_test.py 8.8.8.8 443

        |
        v
+-----------------------+
| 1. DNS Resolution     |
+-----------------------+
        |
        v
+-----------------------+
| 2. ICMP Ping          |
+-----------------------+
        |
        v
+-----------------------+
| 3. TCP/443 Connection |
+-----------------------+
        |
        v
+-----------------------+
| 4. TCP Latency        |
+-----------------------+
        |
        v
+-----------------------+
| 5. Source IP/Port     |
+-----------------------+
        |
        v
+-----------------------+
| 6. Traceroute         |
+-----------------------+
        |
        v
+-----------------------+
| 7. Packet Capture     |
+-----------------------+
        |
        v
     test.pcap
        |
        v
     Wireshark
```

That would give you a reusable troubleshooting utility for testing Cisco/Juniper/Arista management IPs, BGP TCP/179, SSH TCP/22, HTTPS TCP/443, NETCONF TCP/830, and other services in your lab.


In [9]:
import socket
import subprocess
import platform
import time
from datetime import datetime

TARGET = "8.8.8.8"
PORT = 443
TIMEOUT = 3


def resolve_dns(target):
    print("\n=== DNS Resolution ===")

    try:
        ip = socket.gethostbyname(target)
        print(f"Target : {target}")
        print(f"IP     : {ip}")
        return ip

    except socket.gaierror as error:
        print(f"DNS resolution failed: {error}")
        return None


def ping_test(target):
    print("\n=== Ping Test ===")

    system = platform.system().lower()

    if system == "windows":
        command = ["ping", "-n", "4", target]
    else:
        command = ["ping", "-c", "4", target]

    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True
        )

        print(result.stdout)

        if result.returncode == 0:
            print("Ping: SUCCESS")
        else:
            print("Ping: FAILED")

    except Exception as error:
        print(f"Ping error: {error}")


def tcp_test(target, port):
    print("\n=== TCP Connectivity Test ===")

    sock = socket.socket(
        socket.AF_INET,
        socket.SOCK_STREAM
    )

    sock.settimeout(TIMEOUT)

    start_time = time.time()

    try:
        result = sock.connect_ex((target, port))

        elapsed_ms = (time.time() - start_time) * 1000

        if result == 0:
            print(f"TCP/{port}: OPEN")
            print(f"Connection time: {elapsed_ms:.2f} ms")

            local_ip, local_port = sock.getsockname()

            print(f"Source IP   : {local_ip}")
            print(f"Source Port : {local_port}")
            print(f"Destination : {target}:{port}")

            return True

        else:
            print(f"TCP/{port}: CLOSED or FILTERED")
            print(f"Socket error code: {result}")

            return False

    except socket.timeout:
        print(f"TCP/{port}: TIMEOUT")

    except Exception as error:
        print(f"TCP test error: {error}")

    finally:
        sock.close()

    return False


def show_route_info(target):
    print("\n=== Route Information ===")

    system = platform.system().lower()

    if system == "windows":
        command = ["tracert", "-d", target]
    else:
        command = ["traceroute", "-n", target]

    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True
        )

        print(result.stdout)

    except Exception as error:
        print(f"Traceroute error: {error}")


def main():

    print("=" * 60)
    print("PYTHON NETWORK TROUBLESHOOTING TOOL")
    print("=" * 60)

    print(f"Time   : {datetime.now()}")
    print(f"Target : {TARGET}")
    print(f"Port   : {PORT}")

    ip = resolve_dns(TARGET)

    if ip is None:
        return

    ping_test(ip)

    tcp_test(ip, PORT)

    show_route_info(ip)

    print("\n" + "=" * 60)
    print("Troubleshooting completed")
    print("=" * 60)


if __name__ == "__main__":
    main()

PYTHON NETWORK TROUBLESHOOTING TOOL
Time   : 2026-08-26 22:08:13.092808
Target : 8.8.8.8
Port   : 443

=== DNS Resolution ===
Target : 8.8.8.8
IP     : 8.8.8.8

=== Ping Test ===

Pinging 8.8.8.8 with 32 bytes of data:
Request timed out.
Request timed out.
Request timed out.
Request timed out.

Ping statistics for 8.8.8.8:
    Packets: Sent = 4, Received = 0, Lost = 4 (100% loss),

Ping: FAILED

=== TCP Connectivity Test ===
TCP/443: OPEN
Connection time: 27.32 ms
Source IP   : 10.0.9.200
Source Port : 56680
Destination : 8.8.8.8:443

=== Route Information ===

Tracing route to 8.8.8.8 over a maximum of 30 hops

  1     2 ms     1 ms     1 ms  10.0.254.11 
  2     *        *        *     Request timed out.
  3     *        *        *     Request timed out.
  4     *        *        *     Request timed out.
  5     *        *        *     Request timed out.
  6     *        *        *     Request timed out.
  7     *        *        *     Request timed out.
  8     *        *        *  

In [10]:
from scapy.all import sniff, wrpcap


def capture_packets(target, port):

    print("\n=== Packet Capture ===")

    packet_filter = f"host {target} and tcp port {port}"

    print(f"Filter: {packet_filter}")
    print("Capturing 20 packets...")

    packets = sniff(
        filter=packet_filter,
        count=20,
        timeout=20
    )

    filename = f"capture_{target}_{port}.pcap"

    wrpcap(filename, packets)

    print(f"Captured packets: {len(packets)}")
    print(f"Saved to: {filename}")

ModuleNotFoundError: No module named 'scapy'

In [11]:
import socket
import subprocess
import platform
import sys
import time
from datetime import datetime


TIMEOUT = 3


def resolve_target(target):
    print("\n[1] DNS / IP Resolution")

    try:
        ip = socket.gethostbyname(target)
        print(f"Hostname   : {target}")
        print(f"Resolved IP: {ip}")
        return ip

    except socket.gaierror as error:
        print(f"Resolution failed: {error}")
        return None


def ping_test(target):
    print("\n[2] ICMP Ping Test")

    if platform.system().lower() == "windows":
        command = ["ping", "-n", "4", target]
    else:
        command = ["ping", "-c", "4", target]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    print(result.stdout)

    if result.returncode == 0:
        print("RESULT: SUCCESS")
        return True

    print("RESULT: FAILED")
    return False


def tcp_test(target, port):
    print(f"\n[3] TCP/{port} Connectivity Test")

    sock = socket.socket(
        socket.AF_INET,
        socket.SOCK_STREAM
    )

    sock.settimeout(TIMEOUT)

    start = time.perf_counter()

    try:
        result = sock.connect_ex((target, port))

        elapsed = (time.perf_counter() - start) * 1000

        if result == 0:

            source_ip, source_port = sock.getsockname()
            destination_ip, destination_port = sock.getpeername()

            print("RESULT      : OPEN")
            print(f"Latency     : {elapsed:.2f} ms")
            print(f"Source      : {source_ip}:{source_port}")
            print(f"Destination : {destination_ip}:{destination_port}")

            return True

        else:

            print("RESULT : CLOSED / FILTERED")
            print(f"Socket error code: {result}")

            return False

    except socket.timeout:

        print("RESULT : TIMEOUT")
        return False

    except Exception as error:

        print(f"ERROR: {error}")
        return False

    finally:

        sock.close()


def traceroute(target):
    print("\n[4] Traceroute")

    if platform.system().lower() == "windows":

        command = [
            "tracert",
            "-d",
            target
        ]

    else:

        command = [
            "traceroute",
            "-n",
            target
        ]

    try:

        result = subprocess.run(
            command,
            capture_output=True,
            text=True
        )

        print(result.stdout)

    except Exception as error:

        print(f"Traceroute error: {error}")


def packet_capture(target, port):

    print("\n[5] Packet Capture")

    try:

        from scapy.all import sniff, wrpcap

    except ImportError:

        print("Scapy is not installed.")
        print("Install using:")
        print("pip install scapy")
        return

    packet_filter = f"host {target} and tcp port {port}"

    print(f"Capture filter: {packet_filter}")
    print("Capturing for 15 seconds...")

    try:

        packets = sniff(
            filter=packet_filter,
            timeout=15
        )

        filename = f"capture_{target}_{port}.pcap"

        wrpcap(
            filename,
            packets
        )

        print(f"Packets captured : {len(packets)}")
        print(f"PCAP saved       : {filename}")

    except Exception as error:

        print(f"Packet capture error: {error}")
        print()
        print("On Windows:")
        print("1. Install Npcap")
        print("2. Run terminal as Administrator")


def main():

    if len(sys.argv) != 3:

        print()
        print("Usage:")
        print("python network_test.py <target> <port>")
        print()
        print("Examples:")
        print("python network_test.py 8.8.8.8 443")
        print("python network_test.py google.com 443")
        print("python network_test.py 10.0.0.10 22")
        print("python network_test.py 10.0.0.10 179")
        print()

        sys.exit(1)

    target = sys.argv[1]

    try:
        port = int(sys.argv[2])

    except ValueError:

        print("Port must be a number.")
        sys.exit(1)

    print("=" * 65)
    print("PYTHON NETWORK TROUBLESHOOTING TOOL")
    print("=" * 65)

    print(f"Time   : {datetime.now()}")
    print(f"Target : {target}")
    print(f"Port   : {port}")

    ip = resolve_target(target)

    if ip is None:
        sys.exit(1)

    ping_test(ip)

    tcp_status = tcp_test(
        ip,
        port
    )

    traceroute(ip)

    print("\n" + "=" * 65)

    if tcp_status:

        print(
            f"FINAL RESULT: {ip}:{port} is reachable over TCP"
        )

    else:

        print(
            f"FINAL RESULT: Unable to establish TCP/{port}"
        )

    print("=" * 65)

    capture = input(
        "\nDo you want to capture packets? (y/n): "
    )

    if capture.lower() == "y":

        packet_capture(
            ip,
            port
        )


if __name__ == "__main__":
    main()


Usage:
python network_test.py <target> <port>

Examples:
python network_test.py 8.8.8.8 443
python network_test.py google.com 443
python network_test.py 10.0.0.10 22
python network_test.py 10.0.0.10 179



SystemExit: 1

C:\Users\v-yaalam\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\IPython\core\interactiveshell.py:3755: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
